# HydraY NNUE - A3: confronto a budget pieno (768 vs HalfKA)

Runtime → Cambia tipo di runtime → **GPU (T4)**. Esegui le celle in ordine
(Runtime → Esegui tutte) e **non lasciare la sessione inattiva**.

Entrambi i bracci girano nella **stessa sessione**: il setup (Rust + due
compilazioni CUDA) è il costo fisso, ripeterlo in due sessioni è spreco.

**I dati si leggono in streaming da Drive**, non da `/content`: il runtime ha
~66 GB liberi e il dataset intero non ci sta. Costo misurato: Drive via FUSE
regge ~26 MB/s mentre bullet ne consumerebbe ~73, quindi il training diventa
I/O-bound e un run da 40 superbatch passa da ~29 a **~82 minuti**. In cambio
non si taglia il dataset.

Prerequisito: il `.bin` **decompresso** su Drive (vedi cella 4).

I due bracci differiscono **solo** per l'architettura NNUE: `halfka-a1` è
`halfka` con `dev` mergiato dentro, quindi `engine/`, `uci/` e `tt/` sono
identici a `dev`. Non usare il branch `halfka` liscio: reintrodurrebbe fino a
20 Elo di differenza di search, più larga della banda del gate.

In [ ]:
!nvidia-smi
!df -h /content | tail -1

In [ ]:
# Rust toolchain
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal
!~/.cargo/bin/cargo --version

In [ ]:
# I due bracci: dev = 768, halfka-a1 = HalfKA con parità di search
%cd /content
!git clone --depth 1 --branch dev       https://github.com/ThomasGhione/HydraY t768
!git clone --depth 1 --branch halfka-a1 https://github.com/ThomasGhione/HydraY thalfka

In [ ]:
# Dati: si punta al file SU DRIVE, niente copia in /content.
# L'assert e' deliberato: subito dopo un upload il file puo' non essere ancora
# visibile via FUSE, e senza il controllo la variabile resterebbe indefinita
# mandando avanti 82 minuti di training su un path letterale.
from google.colab import drive
drive.mount('/content/drive')
import os, glob
c = glob.glob('/content/drive/MyDrive/**/hydray_v*_shuffled.bin', recursive=True)
assert c, 'dataset non trovato su Drive: upload finito? path giusto?'
DATA = c[0]
# Passato come variabile d'ambiente, non con l'interpolazione {} di IPython:
# quest'ultima, se il nome non esiste, lascia il testo com'e' invece di dare errore.
os.environ['DATA'] = DATA
print('uso:', DATA)
print(os.path.getsize(DATA) / 32, 'posizioni')

In [ ]:
# Braccio 1 - architettura attuale 768. ~82 min (I/O-bound su Drive).
# La loss deve SCENDERE; i checkpoint intermedi ogni 10 SB sono l'assicurazione
# contro la morte della sessione.
%cd /content/t768/nnue/trainer
!PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda cargo run -r --bin trainer --features cuda -- "$DATA" 40 hydray-768-full
!ls checkpoints/
!cp -r checkpoints/hydray-768-full-40 /content/drive/MyDrive/

In [ ]:
# Braccio 2 - HalfKA. Stesso dataset, stesso budget, stessa search.
%cd /content/thalfka/nnue/trainer
!PATH=$HOME/.cargo/bin:$PATH CUDA_PATH=/usr/local/cuda cargo run -r --bin trainer --features cuda -- "$DATA" 40 hydray-halfka-full
!ls checkpoints/
!cp -r checkpoints/hydray-halfka-full-40 /content/drive/MyDrive/

In [ ]:
# Verifica dei due quantised.bin con il lettore di riferimento di ciascun branch.
# Taglie attese: 803.904 B (768) e 3.163.200 B (HalfKA) - la dimensione e' il
# check piu' rapido che l'architettura sia quella giusta.
!ls -l /content/drive/MyDrive/hydray-768-full-40/quantised.bin /content/drive/MyDrive/hydray-halfka-full-40/quantised.bin
%cd /content/t768/nnue/trainer
!PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- /content/drive/MyDrive/hydray-768-full-40/quantised.bin
%cd /content/thalfka/nnue/trainer
!PATH=$HOME/.cargo/bin:$PATH cargo run -r --bin sanity -- /content/drive/MyDrive/hydray-halfka-full-40/quantised.bin

## Cosa guardare

- **`KQvK`**: deve smettere di valere ~0 cp. È il test del fix B1 sul bucket 0.
- **Coppia mirror** (`wN on e4` / `bN on e5`): valori identici.
- **Loss finale** dei due bracci: primo confronto quantitativo, prima dell'SPRT.
- **Taglia dei file**: 803.904 e 3.163.200 byte esatti.

Poi si scaricano i due `quantised.bin` e si prosegue in locale con build,
`nnue-selftest` e SPRT.